In [0]:
# Đọc tất cả các file part trong thư mục Volume
path = "/Volumes/workspace/default/eth_data/"
df_indexed = spark.read.parquet(path)

# Kiểm tra xem đã nhận đủ dữ liệu chưa
print(f"Tổng số dòng dữ liệu: {df_indexed.count()}")
df_indexed.show(20)

Tổng số dòng dữ liệu: 36343432
+------------+----------+--------+------------------+
|from_user_id|to_user_id|token_id|    adjusted_value|
+------------+----------+--------+------------------+
|      330099|         8|   15798| 3.412193104664305|
|      330099|         8|   15798|2.2995448139449395|
|      330099|         8|   15798|2.4812423643806008|
|      330099|         8|   15798|2.7186899383789562|
|      330099|         8|   15798|3.1325192512671918|
|      330099|         8|   15798| 2.818208350049659|
|      330099|         8|   15798| 2.170562770179031|
|      330099|         8|   15798| 2.974483219614471|
|      330099|         8|   15798|1.1151640915528713|
|      330099|         8|   15798| 2.071403342237995|
|      330099|         8|   15798| 2.337093041090904|
|      330099|        23|   15798|               4.6|
|      330099|        23|   15798|              2.29|
|      330099|        36|   15798| 2.027015330780818|
|      330099|        36|   15798|16.80641615642979

In [0]:
import pyspark.sql.functions as F

print("--- Đang chuẩn bị mạng lưới từ 36 triệu giao dịch ---")

# 1. XÂY DỰNG MẠNG LƯỚI (Xóa bỏ hoàn toàn .cache())
edges = df_indexed.select(
    F.col("from_user_id").alias("src"),
    F.col("to_user_id").alias("dst")
).filter("src != dst").distinct()

vertices = edges.select(F.col("src").alias("id")) \
    .union(edges.select(F.col("dst").alias("id"))) \
    .distinct()

# Đếm số lượng ví để khởi động Spark
num_vertices = vertices.count()
print(f"Số lượng nút (Ví) cần xử lý: {num_vertices}")

# 2. THUẬT TOÁN PAGERANK (5 iterations)
print("--- Bắt đầu tính toán PageRank ---")

# Khởi tạo rank ban đầu = 1.0 cho mọi ví
ranks = vertices.withColumn("pagerank", F.lit(1.0))

# Tính Out-degree
out_degrees = edges.groupBy("src").count().withColumnRenamed("count", "out_degree")
edges_with_out_degree = edges.join(out_degrees, "src")

for i in range(5):
    print(f">>> Đang chạy vòng lặp thứ {i+1}/5...")
    
    # Bước 1: Chia điểm uy tín
    contributions = edges_with_out_degree.join(ranks, edges_with_out_degree.src == ranks.id) \
        .select(F.col("dst"), (F.col("pagerank") / F.col("out_degree")).alias("contrib"))
    
    # Bước 2: Thu hoạch điểm và áp dụng Damping Factor (0.85)
    main_ranks = contributions.groupBy("dst").agg(F.sum("contrib").alias("sum_contrib")) \
        .select(F.col("dst").alias("id"), (F.lit(0.15) + F.lit(0.85) * F.col("sum_contrib")).alias("pagerank"))
    
    # Bước 3: Giữ lại những ví không có giao dịch mới (Dangling nodes)
    dangling_ranks = vertices.join(main_ranks, "id", "left_anti") \
        .select(F.col("id"), F.lit(0.15).alias("pagerank"))
    
    # Gộp kết quả
    ranks = main_ranks.union(dangling_ranks).distinct()
    
    # TRÊN SERVERLESS: Ta không dùng localCheckpoint(), cứ để Spark tự tối ưu thực thi
    # Nếu dữ liệu quá lớn, Spark Serverless sẽ tự động "xả" bớt ra đĩa đệm.

# 3. LƯU THÀNH QUẢ VÀO DELTA TABLE
output_table = "workspace.default.pagerank_final_results"
ranks.write.format("delta").mode("overwrite").saveAsTable(output_table)

print(f"\n Kết quả lưu tại: {output_table}")
display(ranks.orderBy(F.desc("pagerank")).limit(20))

--- Đang chuẩn bị mạng lưới từ 36 triệu giao dịch ---
Số lượng nút (Ví) cần xử lý: 4836441
--- Bắt đầu tính toán PageRank ---
>>> Đang chạy vòng lặp thứ 1/5...
>>> Đang chạy vòng lặp thứ 2/5...
>>> Đang chạy vòng lặp thứ 3/5...
>>> Đang chạy vòng lặp thứ 4/5...
>>> Đang chạy vòng lặp thứ 5/5...

 Kết quả lưu tại: workspace.default.pagerank_final_results


id,pagerank
784248,95571.26467159702
516499,40534.32877620213
480915,40491.90835562806
3218106,36569.79058349605
2867319,36162.95397284733
1634001,32794.83964524211
4184876,21761.001470647316
4000287,19699.309131384558
4682412,17683.255399974605
864043,15565.887900923199


In [0]:
# Lấy top 1000 ví có điểm cao nhất
top_whales = ranks.orderBy(F.desc("pagerank")).limit(1000)

# Chuyển thành Pandas để dễ dàng tải về máy local dưới dạng CSV
top_whales_pd = top_whales.toPandas()
top_whales_pd.to_csv("top_1000_whales_pagerank.csv", index=False)

In [0]:
import pyspark.sql.functions as F

# 1. Lấy Top 2000 ví (Cá mập/Sàn) - Nhóm này chắc chắn phải đánh nhãn
top_whales = ranks.orderBy(F.desc("pagerank")).limit(2000)

# 2. Lấy 3000 ví tầm trung (Mid-tier) 
# Chọn những ví có điểm quanh mức trung bình
avg_val = ranks.select(F.avg("pagerank")).collect()[0][0]
mid_fish = ranks.filter(F.col("pagerank").between(avg_val * 0.8, avg_val * 1.2)).limit(3000)

# 3. Lấy 5000 ví điểm thấp (Cá con) nhưng vẫn có hoạt động
small_fish = ranks.orderBy(F.asc("pagerank")).limit(5000)

# Gộp lại thành danh sách 10.000 ví mẫu để Member tầng 2 gán nhãn hành vi
final_sample = top_whales.union(mid_fish).union(small_fish).distinct()

# Xuất ra Pandas để tải CSV
final_sample.toPandas().to_csv("eth_behavior_sample_layer2.csv", index=False)

In [0]:
# 1. Định nghĩa đường dẫn lưu trong Volume (Bạn kiểm tra lại tên Volume của mình nhé)
# Mình giả định đường dẫn giống lúc bạn đọc data: /Volumes/workspace/default/eth_data/
output_path = "/Volumes/workspace/default/eth_data/full_pagerank_results_parquet"

# 2. Đọc bảng từ Catalog
full_ranking = spark.table("workspace.default.pagerank_final_results")

# 3. CHIA NHỎ VÀ LƯU (Ưu tiên Parquet)
# repartition(10) sẽ chia 36tr dòng thành 10 file nhỏ đều nhau
print("--- Đang chia nhỏ và lưu toàn bộ dữ liệu vào Volume ---")

full_ranking.repartition(10).write.mode("overwrite").parquet(output_path)

print(f"Toàn bộ Ranking đã được lưu tại: {output_path}")

--- Đang chia nhỏ và lưu toàn bộ dữ liệu vào Volume ---
Toàn bộ Ranking đã được lưu tại: /Volumes/workspace/default/eth_data/full_pagerank_results_parquet
